# Explicit Quotations and Translation Scores

## tl;dr

No: explicit quotations do not predict better reference-similarity scores in the frozen 100-entry Kappa cohort. The 22 quotation entries have a lower four-metric mean (62.8% versus 75.8%). They are also much longer. A linear word-count adjustment leaves a -6.7 percentage-point association (95% CI -13.1 to -0.3; HC3 p=0.040), but the quotation flag adds little and specification-sensitive out-of-sample predictive value beyond length.

## Context & Methods

The July 23 meeting asked whether a passage containing a quotation from an earlier author might score better because that material could have appeared elsewhere in model training data. This notebook tests the reproducible first-stage feature agreed in the meeting: an explicit quotation span in the edited Greek source.

The outcome is similarity between one `gpt-5.6-sol` v3 translation per entry and the approved house translation. The primary outcome is the unweighted mean of BLEU-4, chrF++, METEOR, and ROUGE-L. The primary regression is score on source word count plus the binary quotation flag. Uncertainty uses HC3 heteroskedasticity-robust standard errors. Incremental prediction is checked with leave-one-out cross-validation.

### Key Assumptions

- An explicit quotation is a balanced `“…”` span in the Greek source. Guillemets around individual letters or forms do not count.
- This detects explicit editorial quotation marking, not uncited paraphrase, parallel passages, or actual inclusion in model training data.
- Reference-similarity metrics are not expert judgments of correctness.
- The checked-in CSV regenerated from the July 23 benchmark is the reviewed fallback because the live PostgreSQL host was unavailable on 2026-07-24.
- The frozen review JSON and scored CSV agree on all 100 headwords and 99 of 100 literal quote-marker flags. The fuller scored source marks Kappa entry 109 as a Dionysius quotation; the clipped review row begins inside the same quoted passage and therefore lacks the opening and closing marks.

In [1]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "generate_kappa_length_quality_page.py").exists():
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root))

from generate_kappa_length_quality_page import (
    OFFICIAL_LIST_PATH,
    analyze_quotation_quality,
    load_csv_rows,
    load_official_entries,
    validate_and_order_rows,
)

input_path = repo_root / "reference_site/statistics/kappa_length_quality.csv"
official_path = repo_root / OFFICIAL_LIST_PATH
source_rows = load_csv_rows(
    input_path,
    profile_name="gpt-5.6-sol",
    profile_version=3,
)
rows = validate_and_order_rows(source_rows, load_official_entries(official_path))
analysis = analyze_quotation_quality(rows)


## Data

The cohort boundary and score coverage are checked before interpreting the comparison.

In [2]:
data_profile = pd.DataFrame(
    [
        {"check": "Official cohort rows", "value": len(rows)},
        {"check": "Unique lemma IDs", "value": len({row["lemma_id"] for row in rows})},
        {"check": "Unique run IDs", "value": len({row["run_id"] for row in rows})},
        {"check": "Explicit-quotation entries", "value": analysis["quote_count"]},
        {"check": "Non-quotation entries", "value": analysis["no_quote_count"]},
        {"check": "Quotation-entry mean Greek words", "value": round(analysis["quote_length_mean"], 1)},
        {"check": "Non-quotation mean Greek words", "value": round(analysis["no_quote_length_mean"], 1)},
        {"check": "Input CSV modified", "value": pd.Timestamp(input_path.stat().st_mtime, unit="s", tz="Australia/Sydney")},
    ]
)
data_profile


,check,value
0,Official cohort rows,100
1,Unique lemma IDs,100
2,Unique run IDs,100
3,Explicit-quotation entries,22
4,Non-quotation entries,78
5,Quotation-entry mean Greek words,61.5
6,Non-quotation mean Greek words,21.7
7,Input CSV modified,2026-07-24 00:26:19.356777430+10:00


## Results

Negative differences mean that quotation entries scored lower. The adjusted effect is the quotation coefficient after a linear source-word-count adjustment.

In [3]:
metric_results = pd.DataFrame(analysis["metric_results"])[
    [
        "metric_label",
        "quote_mean",
        "no_quote_mean",
        "raw_difference",
        "adjusted_effect",
        "ci_95_low",
        "ci_95_high",
        "p_value",
        "baseline_r2",
        "quotation_r2",
        "delta_r2",
        "baseline_mae",
        "quotation_mae",
    ]
]
metric_results.round(4)


,metric_label,quote_mean,no_quote_mean,raw_difference,adjusted_effect,ci_95_low,ci_95_high,p_value,baseline_r2,quotation_r2,delta_r2,baseline_mae,quotation_mae
0,Four-metric mean,0.6284,0.7578,-0.1294,-0.0672,-0.1313,-0.0030,0.0403,0.1868,0.2147,0.0279,0.0874,0.0874
1,BLEU-4,0.4112,0.5953,-0.1841,-0.1004,-0.1937,-0.0071,0.0352,0.1441,0.1723,0.0283,0.1316,0.1328
2,chrF++,0.6627,0.7799,-0.1173,-0.0646,-0.1262,-0.0029,0.0402,0.1775,0.2100,0.0325,0.0783,0.0784
3,METEOR,0.7152,0.8291,-0.1139,-0.0545,-0.1216,0.0125,0.1097,0.1962,0.2083,0.0121,0.0799,0.0808
4,ROUGE-L,0.7244,0.8267,-0.1023,-0.0492,-0.1013,0.0030,0.0642,0.1638,0.1821,0.0183,0.0791,0.0785


In [4]:
length_sensitivity = pd.DataFrame(analysis["length_sensitivity"])[
    [
        "length_form",
        "baseline_r2",
        "quotation_r2",
        "delta_r2",
        "baseline_mae",
        "quotation_mae",
        "delta_mae",
    ]
]
length_sensitivity.round(4)


,length_form,baseline_r2,quotation_r2,delta_r2,baseline_mae,quotation_mae,delta_mae
0,Linear words,0.1868,0.2147,0.0279,0.0874,0.0874,0.0000
1,Log words,0.2825,0.2869,0.0044,0.0811,0.0816,0.0005
2,Quadratic words,0.2289,0.2272,-0.0017,0.0842,0.0846,0.0003


## Takeaways

- The hypothesised positive relationship is not present. Every reported metric moves in the opposite direction in the raw comparison.
- Length explains part, but not all, of the lower scores attached to quotation entries under the linear specification.
- The quotation flag is not a robust operational predictor beyond length: leave-one-out R² changes little, mean absolute error does not improve, and the small R² gain disappears under alternative length specifications.
- A separate manual annotation of paraphrases and parallel passages is still needed to test the broader training-exposure hypothesis.